# CUDA dev notebook — Sensor Anomaly Detector

Runtime: Runtime > Change runtime type > GPU (T4 is fine for now).

Workflow: write/compile/run kernels here, then copy finished `.cu`/`.py` files back into the GitHub repo locally and commit from your machine.

In [ ]:
!nvcc --version
!nvidia-smi

## Clone the repo (so kernels stay versioned)

In [ ]:
!git clone https://github.com/nrwre/cudaproject.git
%cd cudaproject

## Compile + run: vector_add sanity check

In [ ]:
!nvcc -O2 cuda/vector_add.cu -o vector_add
!./vector_add

## Week 2: numeric validation

Generate one synthetic dataset, feed the *same* file to the CUDA kernel and the NumPy reference, then diff their anomaly flags exactly. This is the only way to actually prove correctness — comparing separately-generated random data proves nothing.

In [ ]:
!python python/generate_data.py data.bin

In [ ]:
!nvcc -O2 cuda/rolling_zscore.cu -o rolling_zscore
!./rolling_zscore data.bin gpu_anomalies.bin

In [ ]:
!python python/cpu_baseline.py data.bin cpu_anomalies.bin

In [ ]:
!python python/validate.py data.bin cpu_anomalies.bin gpu_anomalies.bin

Expect 0 mismatches (or a tiny number from float32 summation-order differences right at the threshold boundary — if that happens, it's worth understanding why, not just accepting it).

## Week 2: memory access pattern (sensor-major vs time-major)

Same algorithm, same thread-per-sensor mapping. v1 reads are strided across a warp (uncoalesced); v2 transposes to time-major so a warp's simultaneous reads are contiguous (coalesced). Compare the kernel-time lines in stderr output above/below.

In [ ]:
!nvcc -O2 cuda/rolling_zscore_coalesced.cu -o rolling_zscore_coalesced
!./rolling_zscore_coalesced data.bin gpu_anomalies_coalesced.bin

In [ ]:
# Validate the coalesced kernel against the same NumPy reference, then compare timings.
!python python/validate.py data.bin cpu_anomalies.bin gpu_anomalies_coalesced.bin

## Bigger dataset (the gap should widen with more sensors)

Memory-bound kernels show the coalescing gap more clearly at scale. Try generating a larger dataset and re-running both kernels to see how the timing gap changes.

In [ ]:
import sys
sys.path.insert(0, 'python')
from generate_data import generate, write_input_file

data, window, threshold = generate(num_sensors=65536, num_timesteps=2048)
write_input_file('data_large.bin', data, window, threshold)
print('wrote data_large.bin')

In [ ]:
!./rolling_zscore data_large.bin gpu_anomalies_large.bin
!./rolling_zscore_coalesced data_large.bin gpu_anomalies_large_coalesced.bin

## Pull changes back

Download any `.cu`/`.py` files you edited here (left sidebar > file browser > right-click > Download), overwrite them locally, then commit + push from your machine as usual.